In [ ]:
import boto3
import pandas as pd
import xgboost as xgb
from pathlib import Path
import sys
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
import joblib
from io import BytesIO

import tarfile

In [2]:
# Make repository root importable when running this file directly.
REPO_ROOT = Path.cwd().parents[1] 
if str(REPO_ROOT) not in sys.path:
    sys.path.append(str(REPO_ROOT))

from data_contracts import (
    HISTORICAL_CATEGORICAL_FEATURES,
    HISTORICAL_INTEGER_FEATURES,
    HISTORICAL_DATE_FEATURES,
    TRAINING_CATEGORICAL_FEATURES,
    TRAINING_INTEGER_FEATURES,
    TRAINING_DATE_FEATURES,
    build_historical_data,
)
from src.services.data_srv import DataService
from project_paths import PATHS, s3_uri

In [3]:
# Initialize S3 client
s3_client = boto3.client('s3')

In [4]:
print(PATHS.bucket_name, PATHS.training_data_key)

sagemaker-us-east-1-719805443631 oead-prob/data/students_prob_show_up_to_lc_data.csv


In [5]:
# Download data from S3 to memory
data_buffer = BytesIO()
s3_client.download_fileobj(PATHS.bucket_name, PATHS.training_data_key, data_buffer)
data_buffer.seek(0)

df = pd.read_csv(data_buffer)

In [6]:
target_column = 'J'

# Enforce data contract for model inputs.
historical_data = build_historical_data(df, target_column=target_column)

In [7]:
service = DataService()
training_data = service.historical_to_training(historical_data)

In [8]:
X = training_data.X
y = training_data.y

In [9]:
# Split the data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [10]:
# Create preprocessing pipeline
preprocessor = ColumnTransformer(
    transformers=[
        ('num', 'passthrough', TRAINING_INTEGER_FEATURES),
        ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), TRAINING_CATEGORICAL_FEATURES),
    ]
)

In [11]:
# Create XGBoost model
model = xgb.XGBClassifier(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    random_state=42
)

In [12]:
# Create pipeline with preprocessing and model
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', model)
])

In [13]:
# Train the pipeline
pipeline.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num', 'passthrough',
                                                  ['ageGroup', 'deltaDays',
                                                   'deltaHours', 'hourOfDay',
                                                   'minuteOfHour', 'isWeekend',
                                                   'is_holiday',
                                                   'is_holiday_pre',
                                                   'is_holiday_post']),
                                                 ('cat',
                                                  OneHotEncoder(drop='first',
                                                                handle_unknown='ignore'),
                                                  ['dow', 'studLevel', 'stuH',
                                                   'country_iso', 'enrollment',
                                                   'native_language',
                                                   'class_ty...
                               feature_types=None, gamma=None, grow_policy=None,
                               importance_type=None,
                               interaction_constraints=None, learning_rate=0.1,
                               max_bin=None, max_cat_threshold=None,
                               max_cat_to_onehot=None, max_delta_step=None,
                               max_depth=6, max_leaves=None,
                               min_child_weight=None, missing=nan,
                               monotone_constraints=None, multi_strategy=None,
                               n_estimators=100, n_jobs=None,
                               num_parallel_tree=None, random_state=42, ...))])

In [14]:
# Evaluate the model
y_pred = pipeline.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f"Model Accuracy: {accuracy:.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Model Accuracy: 0.7509

Classification Report:
              precision    recall  f1-score   support

           0       0.66      0.39      0.49     15124
           1       0.77      0.91      0.84     34005

    accuracy                           0.75     49129
   macro avg       0.72      0.65      0.66     49129
weighted avg       0.74      0.75      0.73     49129



In [16]:
# 1️⃣ Serialize pipeline into memory
model_buffer = BytesIO()
joblib.dump(pipeline, model_buffer)
model_buffer.seek(0)

# 2️⃣ Create tar.gz archive in memory
tar_buffer = BytesIO()

with tarfile.open(fileobj=tar_buffer, mode="w:gz") as tar:
    # Create TarInfo object for model.pkl
    tarinfo = tarfile.TarInfo(name="model.pkl")
    tarinfo.size = len(model_buffer.getvalue())

    # Reset model buffer pointer before adding
    model_buffer.seek(0)

    tar.addfile(tarinfo=tarinfo, fileobj=model_buffer)

# Important: reset pointer before upload
tar_buffer.seek(0)

# 3️⃣ Upload tar.gz to S3
s3_client.upload_fileobj(
    tar_buffer,
    PATHS.bucket_name,
    PATHS.pipeline_output_key  # should end with model.tar.gz
)

print(f"Pipeline packaged and uploaded to S3: {s3_uri(PATHS.pipeline_output_key)}")

Pipeline packaged and uploaded to S3: s3://sagemaker-us-east-1-719805443631/oead-prob/models/model.tar.gz


In [15]:
# # Save pipeline to memory buffer
# pipeline_buffer = BytesIO()
# joblib.dump(pipeline, pipeline_buffer)
# pipeline_buffer.seek(0)

# # Upload pipeline to S3 from memory
# s3_client.upload_fileobj(pipeline_buffer, PATHS.bucket_name, PATHS.pipeline_output_key)

# print(f"Pipeline saved to S3: {s3_uri(PATHS.pipeline_output_key)}")